# 🚀 Тестирование Hunyuan3D-2.1 (Optimized for 8GB)
## 🎯 Цель ноутбука

Основная цель — оценка производительности и качества модели **Hunyuan3D-2.1** в условиях ограниченных ресурсов (RTX 4060, 8GB VRAM) под управлением Windows.

**Задачи:**
1. Проверить стабильность работы модели.
2. Оценить качество PBR-материалов и плотность генерируемого меша.
3. Сравнить результаты с другими моделями.


## 🛠 1. Настройка окружения

In [1]:
import os
import sys
import torch
import gc
import glob
import time
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import display, HTML

NOTEBOOK_DIR = os.getcwd()
BASE_ML = os.path.abspath(os.path.join(NOTEBOOK_DIR, "..", ".."))
H3D2_PATH = os.path.join(BASE_ML, "H3D2")
TRELLIS_PATH = os.path.join(BASE_ML, "TRELLIS2")
OUTPUT_BASE_DIR = os.path.join(NOTEBOOK_DIR, "output")
SRC_IMAGES_DIR = os.path.join(BASE_ML, "src")

if H3D2_PATH not in sys.path:
    sys.path.insert(0, H3D2_PATH)
    sys.path.insert(0, os.path.join(H3D2_PATH, "hy3dshape"))
    sys.path.insert(0, os.path.join(H3D2_PATH, "hy3dpaint"))

if TRELLIS_PATH not in sys.path:
    sys.path.insert(0, TRELLIS_PATH)

os.chdir(H3D2_PATH)

from hy3dshape.pipelines import Hunyuan3DDiTFlowMatchingPipeline
from textureGenPipeline import Hunyuan3DPaintPipeline, Hunyuan3DPaintConfig
from trellis2.pipelines.rembg.BiRefNet import BiRefNet

def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

print(f"Готов к работе. Проект: {H3D2_PATH}")
clear_gpu()

FlashAttention is not available.
Готов к работе. Проект: C:\Develop\diploma\Machine_learning\H3D2


## 🧠 2. Инициализация моделей (Low VRAM Mode)

In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_PATH = 'tencent/Hunyuan3D-2.1'

print("Загрузка Shape Pipeline (DiT)...")
pipeline_shapegen = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(MODEL_PATH)

# КРИТИЧЕСКАЯ ОПТИМИЗАЦИЯ ДЛЯ 8GB: Автоматический поочередный перенос моделей
if hasattr(pipeline_shapegen, 'enable_model_cpu_offload'):
    pipeline_shapegen.enable_model_cpu_offload()
    print("✅ Режим enable_model_cpu_offload включен")
else:
    pipeline_shapegen.to(DEVICE)

print("Загрузка Paint Pipeline (PBR)...")
conf = Hunyuan3DPaintConfig(max_num_view=6, resolution=512)
conf.realesrgan_ckpt_path = os.path.join(H3D2_PATH, "hy3dpaint/ckpt/RealESRGAN_x4plus.pth")
conf.multiview_cfg_path = os.path.join(H3D2_PATH, "hy3dpaint/cfgs/hunyuan-paint-pbr.yaml")
conf.custom_pipeline = os.path.join(H3D2_PATH, "hy3dpaint/hunyuanpaintpbr")
paint_pipeline = Hunyuan3DPaintPipeline(conf)

print("Загрузка BiRefNet...")
remover = BiRefNet(model_name="briaai/RMBG-2.0")
remover.to(DEVICE)

print("Модели загружены. Оптимизации применены.")
clear_gpu()

2026-05-05 17:38:10,086 - hy3dgen.shapgen - INFO - Try to load model from local path: C:\Users\Vlad/.cache/hy3dgen\tencent/Hunyuan3D-2.1\hunyuan3d-dit-v2-1
2026-05-05 17:38:10,093 - hy3dgen.shapgen - INFO - Loading model from C:\Users\Vlad/.cache/hy3dgen\tencent/Hunyuan3D-2.1\hunyuan3d-dit-v2-1\model.fp16.ckpt


Загрузка Shape Pipeline (DiT)...
using moe
using moe
using moe
using moe
using moe
using moe
PointCrossAttentionEncoder INFO: pc_sharpedge_size is zero
✅ Режим enable_model_cpu_offload включен
Загрузка Paint Pipeline (PBR)...
Загрузка BiRefNet...
[INFO] Local RMBG-2.0 found. Loading from: C:\Develop\diploma\Machine_learning\TRELLIS2\MODELS\RMBG-2.0
Модели загружены. Оптимизации применены.


## 🚀 3. Запуск генерации

In [3]:
def run_generation(image_path):
    file_name = os.path.basename(image_path)
    output_name = os.path.splitext(file_name)[0]
    run_output_dir = os.path.join(OUTPUT_BASE_DIR, output_name)
    os.makedirs(run_output_dir, exist_ok=True)
    
    print(f"--- Обработка {file_name} ---")
    
    # 1. Удаление фона
    input_img = Image.open(image_path).convert("RGB")
    rgba_img = remover(input_img)
    rgba_img.save(os.path.join(run_output_dir, "rgba.png"))
    clear_gpu()
    
    # 2. Генерация геометрии
    print(f"Генерация меша...")
    # Теперь нам не нужно делать .to(DEVICE) вручную, пайплайн сам это делает
    mesh = pipeline_shapegen(
        image=rgba_img, 
        num_inference_steps=20, # Снижаем до 20 для максимизации скорости
        guidance_scale=5.0
    )[0]
    
    mesh_path = os.path.join(run_output_dir, "shape.glb")
    mesh.export(mesh_path)
    clear_gpu()
    
    # 3. Текстурирование
    print(f"Текстурирование...")
    output_mesh_path = os.path.join(run_output_dir, "textured.glb")
    paint_pipeline(
        mesh_path = mesh_path, 
        image_path = image_path,
        output_mesh_path = output_mesh_path
    )
    
    print(f"✅ {file_name} успешно обработан")
    clear_gpu()

image_files = []
for ext in ['*.png', '*.jpg', '*.jpeg', '*.webp']:
    image_files.extend(glob.glob(os.path.join(SRC_IMAGES_DIR, ext)))

for img_path in sorted(image_files):
    try:
        start_time = time.time()
        run_generation(img_path)
        print(f"Время выполнения: {time.time() - start_time:.2f} сек.")
    except Exception as e:
        print(f"❌ Ошибка: {e}")
        clear_gpu()

--- Обработка building.png ---
Генерация меша...
[PROFILE] Moving Cond + Model to GPU: 0.0046s


Diffusion Sampling::   0%|                                                                      | 0/20 [01:43<?, ?it/s]


❌ Ошибка: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!
--- Обработка feature_chair.png ---
Генерация меша...
[PROFILE] Moving Cond + Model to GPU: 0.8362s


Diffusion Sampling::   0%|                                                                      | 0/20 [01:01<?, ?it/s]


❌ Ошибка: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!
--- Обработка forest.jpg ---
Генерация меша...
[PROFILE] Moving Cond + Model to GPU: 0.5372s


...

❌ Ошибка: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!
--- Обработка test.jpg ---
Генерация меша...
[PROFILE] Moving Cond + Model to GPU: 0.5537s


Diffusion Sampling::   0%|                                                                      | 0/20 [00:02<?, ?it/s]


❌ Ошибка: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!
--- Обработка vlad_T.png ---
Генерация меша...
[PROFILE] Moving Cond + Model to GPU: 0.5431s


Diffusion Sampling::   0%|                                                                      | 0/20 [00:02<?, ?it/s]


❌ Ошибка: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!


## Результат

Не удалось запустить на 4060 с 8gb vram